<a href="https://colab.research.google.com/github/simonlim563/asl-ml-immersion/blob/master/custom_ai_cross_video/clients/VN_Zott/optimization_zott_vn_12082025_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def prepare_notebook():

    """    This function pulls the .py files from cloud storage bucket and installs the necessary dependancies    """
    import os # Create a directory to store the copied files
    from datetime import datetime, timedelta
    os.makedirs('executor_files', exist_ok=True)# Use gsutil to copy the .py files

    # Generate yesterday's date string (YYYYMMDD)
    yesterday_str = (datetime.today() - timedelta(days=1)).strftime("%Y%m%d")

    # Build the GCS path for yesterday's historical adjustments file
    historical_file_gcs = f"gs://apac-github-test/Files_for_vertex_ai/Cross_Video_Zott_historical/historical_adjustments_{yesterday_str}.parquet"

    !gsutil -m cp gs://apac-github-test/Files_for_vertex_ai/mp-adh-groupm-sg-cea49b7167eb.json executor_files/
    !gsutil -m cp gs://apac-github-test/Files_for_vertex_ai/*.py executor_files/
    !gsutil -m cp gs://apac-github-test/Files_for_vertex_ai/Cross_Video_Zott_historical/historical_adjustments_20250812.parquet executor_files/

    # Add the directory to your Python path
    import sys
    sys.path.append('executor_files')

    !pip install requests
    !pip install pandas
    !pip install numpy
    !pip install scipy
    !pip install six
    !pip install google-cloud-secret-manager
    !pip install google-cloud-bigquery
    !pip install google-api-python-client
    !pip install google-auth
    !pip install google-auth-httplib2
    !pip install google-api-core
    !pip install google-cloud-storage
    !pip install db-dtypes

    import MS_DV360API_Connector

    return print("Everything has been installed successfully - notebook good to go!")

prepare_notebook()

In [ ]:
import json
import pandas as pd
from datetime import date, timedelta, datetime
import numpy as np
from scipy.optimize import minimize
import requests
from contextlib import closing
from six.moves.urllib.request import urlopen
from googleapiclient import discovery
from io import StringIO
from googleapiclient.discovery import build
from google.oauth2 import service_account
from googleapiclient import http as googleHttp
import time
import random
import os
import db_dtypes
from MS_DV360API_Connector import DV360Connector
from urllib.parse import urlencode
from six.moves.urllib.parse import urlencode, urlunparse
from six import string_types
import io
import datetime as dt
from google.cloud import storage


# import seaborn as sns
# import matplotlib.pyplot as plt

### Load DV360 report

In [ ]:
current_date = datetime.now()

# Set up the parameters to request an API report

json_file_path = "executor_files/mp-adh-groupm-sg-cea49b7167eb.json"

if not os.path.exists(json_file_path):
    print(f"Error: JSON file not found at {json_file_path}")
    exit(1)

try:
    with open(json_file_path, 'r', encoding='utf-8') as f:
        config_data = json.load(f)
    print(config_data)

except FileNotFoundError:
    print(f"Error: JSON file not found at {json_file_path}")
    exit(1)

except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}")
    exit(1)

except Exception as e:
    print(f"An unexpected error occurred while loading JSON: {e}")
    exit(1)

dv360_connector = DV360Connector('executor_files/mp-adh-groupm-sg-cea49b7167eb.json')
io_id = ['1022529463', '1022173941']    ## This is actually IO ID
start_date = '2025-07-31'   ## The report start date
end_date = '2025-08-31'

print(f"The report will include activity from {start_date} to {end_date}")

## Below is the list of categorical columns and metrics the report will include (divided by columns and metrics)
## You can find more elements and its Console equivalent here https://developers.google.com/bid-manager/v1.1/filters-metrics

columns_list = ['FILTER_DATE',
                'FILTER_PARTNER_NAME',
                'FILTER_PARTNER',
                'FILTER_COUNTRY',
                 'FILTER_ADVERTISER_CURRENCY',
                'FILTER_ADVERTISER_NAME',
                 'FILTER_ADVERTISER',
                'FILTER_INSERTION_ORDER_NAME',
                 'FILTER_INSERTION_ORDER',
                'FILTER_LINE_ITEM_NAME',
                 'FILTER_LINE_ITEM',
                #  'FILTER_DEVICE_TYPE',
                # 'FILTER_PAGE_LAYOUT'
                 ]

metrics_list = ['METRIC_IMPRESSIONS','METRIC_TRUEVIEW_VIEWS', 'METRIC_RICH_MEDIA_VIDEO_PLAYS', 'METRIC_RICH_MEDIA_VIDEO_COMPLETIONS','METRIC_TOTAL_MEDIA_COST_ADVERTISER','METRIC_REVENUE_ADVERTISER']

## CODE TO RUN THE API REPORT BELOW

report_definition = dv360_connector.build_report_definition(metrics_list, io_id, columns_list, start_date, end_date)
report_raw = dv360_connector.download_report(report_definition)


In [ ]:
report_df = report_raw.iloc[:-3]

In [ ]:
report_df.tail()

### Load TikTok report

In [ ]:
ACCESS_TOKEN = "d118cb774b86a7eae6a5a06a1164e8573efa264c"

In [ ]:
ADVERTISER_ID = [7371003546684424193]  # INPUT THE ADVERTISER ID directly from Tiktok
START_DATE = date(2025, 7, 31)
END_DATE = date(2025, 8, 31)

In [ ]:
# Set the metrics you want to pull from the report

SERVICE_TYPE = "AUCTION"
REPORT_TYPE = "BASIC"
DATA_LEVEL = "AUCTION_AD"
DIMENSIONS = ["ad_id", "stat_time_day"]
METRICS = [
    #Basic data metrics
    "impressions",
    "clicks",
    "total_landing_page_view",
    "conversion",
    "result",
    "secondary_goal_result",
    "spend",

    # Video play metrics
     "video_play_actions",
    # "video_watched_2s",
    # "video_watched_6s",
    # "average_video_play",
    # "video_views_p25",
    # "video_views_p50",
    # "video_views_p75",
    # "video_views_p100",
    # "engaged_view",
     "engaged_view_15s",


    # Basic data metrics - Reach
    # "reach",
    # "frequency",

    #Engagement metrics
    # "engagements",
    # "profile_visits",
    # "likes",
    # "comments",
    # "shares",
    # "follows",
    # "anchor_clicks",

    #ROAS metrics
    # "onsite_shopping",
    # "cost_per_onsite_shopping",
    # "onsite_shopping_rate",
    # "value_per_onsite_shopping",
    # "total_onsite_shopping_value",



    # attributes
    "currency",
    "campaign_id",
    "adgroup_id",
    "campaign_budget",
    "objective_type",
    "budget",
    "billing_event",
    "bid_strategy",
    "bid",
    #"split_test"
]


PAGE_SIZE = 1000
QUERY_MODE = 'CHUNK'

In [ ]:
# API Query for pulling the dimension keys and metrics

PATH = "/open_api/v1.3/report/task/create/"


def build_url(path, query=""):
    # type: (str, str) -> str
    """
    Build request URL
    :param path: Request path
    :param query: Querystring
    :return: Request URL
    """
    scheme, netloc = "https", "business-api.tiktok.com"
    return urlunparse((scheme, netloc, path, "", query, ""))


def post(json_str):
    # type: (str) -> dict
    """
    Send GET request
    :param json_str: Args in JSON format
    :return: Response in JSON format
    """
    args = json.loads(json_str)
    query_string = urlencode(
        {
            k: v if isinstance(v, string_types) else json.dumps(v)
            for k, v in args.items()
        }
    )
    url = build_url(PATH, query_string)
    headers = {"Access-Token": ACCESS_TOKEN}
    rsp = requests.post(url, headers=headers)
    return rsp.json()


def batch_iterator(ids, batch_size):
    for i in range(0, len(ids), batch_size):
        yield ids[i : i + batch_size]


if __name__ == "__main__":

    service_type = SERVICE_TYPE
    report_type = REPORT_TYPE
    data_level = DATA_LEVEL
    dimensions_list = DIMENSIONS
    dimensions = json.dumps(dimensions_list)
    metrics_list = METRICS
    metrics = json.dumps(metrics_list)
    start_date = START_DATE.strftime("%Y-%m-%d")
    end_date = END_DATE.strftime("%Y-%m-%d")
    page_size = PAGE_SIZE
    advertiser_task_map = {}
    task_unsuccessful = {}

    # query_mode = QUERY_MODE

    data_df = pd.DataFrame()
    # Args in JSON format

    for index, advertiser_id in enumerate(ADVERTISER_ID):
        # my_args = "{\"metrics\": %s, \"data_level\": \"%s\", \"end_date\": \"%s\", \"page_size\": \"%s\", \"start_date\": \"%s\", \"advertiser_id\": \"%s\",  \"report_type\": \"%s\", \"dimensions\": %s, \"service_type\": \"%s\" }" % (metrics, data_level, end_date, page_size, start_date, advertiser_id, report_type, dimensions, service_type)
        my_args = (
            '{"advertiser_id": "%s", "service_type": "%s", "report_type": "%s",  "data_level": "%s", "dimensions": %s, "metrics": %s, "start_date": "%s", "end_date": "%s", "page_size": %s, "enable_report_title_translation": false }'
            % (
                advertiser_id,
                service_type,
                report_type,
                data_level,
                dimensions,
                metrics,
                start_date,
                end_date,
                page_size,
            )
        )

        result = post(my_args)
        #print(result)
        if result["code"] == 0:
            task_id = result["data"]["task_id"]
            advertiser_task_map[advertiser_id] = task_id
            # print(advertiser_task_map)
        else:
            message = result["message"]
            task_unsuccessful[advertiser_id] = message

        if index > 0 and index % 9 == 0:
            time.sleep(15)
        else:
            time.sleep(3)

    # for advertiser_id, message in task_unsuccessful:
    #     print(f"task creation was unsuccessful for {advertiser_id} due to {message}")




In [ ]:
time.sleep(300)

In [ ]:


PATH_CHECK = "/open_api/v1.3/report/task/check/"
PATH_DOWNLOAD = "/open_api/v1.3/report/task/download/"


def build_url(path, query=""):
    # type: (str, str) -> str
    """
    Build request URL
    :param path: Request path
    :param query: Querystring
    :return: Request URL
    """
    scheme, netloc = "https", "business-api.tiktok.com"
    return urlunparse((scheme, netloc, path, "", query, ""))


def check(json_str):
    # type: (str) -> dict
    """
    Send GET request
    :param json_str: Args in JSON format
    :return: Response in JSON format
    """
    args = json.loads(json_str)
    query_string = urlencode(
        {
            k: v if isinstance(v, string_types) else json.dumps(v)
            for k, v in args.items()
        }
    )
    url = build_url(PATH_CHECK, query_string)
    headers = {
        "Access-Token": ACCESS_TOKEN,
    }
    rsp = requests.get(url, headers=headers)
    return rsp.json()


def download(json_str):
    # type: (str) -> dict
    """
    Send GET request
    :param json_str: Args in JSON format
    :return: Response in JSON format
    """
    args = json.loads(json_str)
    query_string = urlencode(
        {
            k: v if isinstance(v, string_types) else json.dumps(v)
            for k, v in args.items()
        }
    )
    url = build_url(PATH_DOWNLOAD, query_string)
    headers = {
        "Access-Token": ACCESS_TOKEN,
    }
    rsp = requests.get(url, headers=headers)
    return rsp.text


if __name__ == "__main__":
    load_df = pd.DataFrame()
    not_ready = {}

    for advertiser_id, task_id in advertiser_task_map.items():

        # Args in JSON format
        my_args = '{"advertiser_id": "%s", "task_id": "%s"}' % (advertiser_id, task_id)
        check_task = check(my_args)
        #print(check_task)

        if check_task["code"] == 0 and check_task["data"]["status"] == "SUCCESS":

            csv_string = download(my_args)
            temp_df = pd.read_csv(io.StringIO(csv_string))
            temp_df["Advertiser ID"] = advertiser_id
            if not temp_df.empty:
                load_df = pd.concat([temp_df, load_df])

        else:
            print(check_task)
            # print(
            #     "The taskid: {} for advertiser {} is not ready yet".format(
            #         task_id, advertiser_id
            #     )
            # )
            # not_ready[advertiser_id] = task_id

        time.sleep(5)

In [ ]:
load_df.tail()

In [ ]:
# prompt: from load_df, filter for rows that have 1839054926886305,1839054793361409 as campaign_id. reset index

filtered_df = load_df[load_df['campaign_id'].isin([1839054926886305, 1839054793361409])]
filtered_df = filtered_df.reset_index(drop=True)
filtered_df.head()


In [ ]:
pd.set_option('display.float_format', lambda x: f'{x:.6f}' if abs(x) < 1e6 else f'{x:.0f}')

### Import historical data

In [ ]:
def clean_adgroup_id(series):
  series = series.astype(str).str.strip()
  series = series.str.replace(r'\.0$', '', regex=True)
  series = series.apply(lambda x: int(float(x)))
  return series.astype('int64')

In [ ]:
yesterday_str = (datetime.today() - timedelta(days=1)).strftime("%Y%m%d")

# Read CSV
historical_df = pd.read_parquet(f"historical_adjustments_{yesterday_str}.parquet")

# Drop NaNs
historical_df = historical_df.dropna(subset=['adgroup_id'])

# Clean adgroup_id
historical_df['adgroup_id'] = clean_adgroup_id(historical_df['adgroup_id'])

# Parse date safely
historical_df['date'] = pd.to_datetime(historical_df['date'], errors='coerce')

# Force entire column to be datetime64[ns] to avoid mixed types
historical_df['date'] = pd.to_datetime(historical_df['date'])

# Rename for consistency
historical_df = historical_df.rename(columns={'platform_total_budget': 'total_budget'})

# Debug check
print(historical_df['date'].dtype)  # should be datetime64[ns]
print(historical_df['date'].unique())

In [ ]:
historical_df.tail(11)

In [ ]:
historical_df.info()

### Manual analysis of performance

#### DV360

In [ ]:
# prompt: group report_df by 'Insertion Order ID'. Keep only 'Insertion Order', 'Insertion Order ID', 'Impressions', 'Starts (Video)', 'Complete Views (Video)' and 'Revenue (Adv Currency)' and store it as a different df. Calculate new metric vcr which is calculated by dividing completed views by starts. Also calculate cpm

# Keep only specified columns
report_df_grouped = report_df[[
    'Insertion Order ID',
    'Insertion Order',
    'Impressions',
    'Starts (Video)',
    'Complete Views (Video)',
    'Revenue (Adv Currency)'
]].copy()

# Group by 'Insertion Order ID' and sum relevant metrics
report_df_grouped = report_df_grouped.groupby('Insertion Order ID').sum().reset_index()

# Calculate VCR
report_df_grouped['VCR'] = (
    report_df_grouped['Complete Views (Video)'] /
    report_df_grouped['Impressions']
).fillna(0)  # Handle potential division by zero

# Calculate CPM
report_df_grouped['CPM'] = (
    report_df_grouped['Revenue (Adv Currency)'] /
    report_df_grouped['Impressions'] * 1000
).fillna(0) # Handle potential division by zero

report_df_grouped


In [ ]:
# Mappings the old line items into the new one
id_mapping = {
 22853599586: 22871714861,  # CTV
 22852881533: 22875518857,  # Computer
 22852881101: 22871714300,  # Smartphone
 22857485470: 22875518767   # Tablet
}

name_mapping = {
 'YT_Category Buyer_Fe18-34_G2_CTV': 'YT_Category Buyer_Fe18-34_G2_CTV_new',
 'YT_Category Buyer_Fe18-34_G2_Computer': 'YT_Category Buyer_Fe18-34_G2_Computer_new',
 'YT_Category Buyer_Fe18-34_G2_Smartphone': 'YT_Category Buyer_Fe18-34_G2_Smartphone_new',
 'YT_Category Buyer_Fe18-34_G2_Tablet': 'YT_Category Buyer_Fe18-34_G2_Tablet_new'
}

# Replace line_item_id and line_item_name using the mappings
report_df['Line Item ID'] = report_df['Line Item ID'].replace(id_mapping)
report_df['Line Item'] = report_df['Line Item'].replace(name_mapping)

In [ ]:
# prompt: group report_df by 'Line Item ID'. Keep only 'Insertion Order ID', 'Line Item ID', 'Line Item', 'Impressions', 'Starts (Video)', 'Complete Views (Video)' and 'Revenue (Adv Currency)' and store it as a different df. Calculate new metric vcr which is calculated by dividing completed views by starts. Also calculate cpm

# Keep only specified columns and group by 'Line Item ID'
report_df_grouped = report_df[[
    'Insertion Order ID',
    'Line Item ID',
    'Line Item',
    'Impressions',
    'Starts (Video)',
    'Complete Views (Video)',
    'Revenue (Adv Currency)'
]].copy()

report_df_grouped = report_df_grouped.groupby('Line Item ID').sum().reset_index()

# Calculate VCR
report_df_grouped['VCR'] = (
    report_df_grouped['Complete Views (Video)'] /
    report_df_grouped['Impressions']
).fillna(0)  # Handle potential division by zero

# Calculate CPM
report_df_grouped['CPM'] = (
    report_df_grouped['Revenue (Adv Currency)'] /
    report_df_grouped['Impressions'] * 1000
).fillna(0) # Handle potential division by zero

report_df_grouped

In [ ]:
#dv360 report
report_df.head(1)

In [ ]:
#tiktok report
filtered_df.head(1)

### Initialising platform ids

In [ ]:
#initialise the platform ids
dv_adgroups = report_df['Line Item ID'].unique().astype(int).tolist()
tiktok_adgroups = filtered_df['adgroup_id'].unique().astype(int).tolist()
tiktok_adgroup_kol = [1839057950747698,1839054793361425]
tiktok_adgroup_nuclass= [1839058353004833,1839054926886321]
dv_adgroup_id_video = [22853603024,22852881341,22853602793]
dv_adgroup_id_yt = [22871714861,22875518857,22871714300,22875518767]
dv_adgroups_1 = dv_adgroup_id_yt +dv_adgroup_id_video
print(dv_adgroups)
print(tiktok_adgroups)
print(dv_adgroups_1)

### Initialising campaign details

In [ ]:
Overall_total_budget_dv = 198000000
Overall_total_budget_tt = 198000000
total_base_budget_tt = 138600000
total_base_budget_dv =  148500000
dv_base_yt_budget = 148500000 * (120000000/(120000000 + 78000000))
dv_base_video_budget = 148500000 * (78000000/(120000000 + 78000000))
floating_budget = Overall_total_budget_dv + Overall_total_budget_tt - total_base_budget_dv - total_base_budget_tt
print('Total overall floating budget :', floating_budget)
print('DV360 YT base budget :',dv_base_yt_budget)
print('DV360 Video base budget:',dv_base_video_budget)
Combined_budget = Overall_total_budget_dv + Overall_total_budget_tt
print(dv_base_yt_budget)

In [ ]:
Percentage_TikTok_base_budget = total_base_budget_tt/Overall_total_budget_tt
Percentage_DV360_base_budget = total_base_budget_dv/Overall_total_budget_dv
print('Percentage of base budget for TIKTOK',Percentage_TikTok_base_budget)
print('Percentage of base budget for DV360',Percentage_DV360_base_budget)

In [ ]:
Total_budget = floating_budget
Start_date = dt.date(2025,7,31)
End_date = dt.date(2025,9,30)
today = date.today()

### Data processing to split float and spend budget

In [ ]:
#Converting date data types
report_df.loc[:, 'Date'] = pd.to_datetime(report_df['Date']).dt.date
filtered_df.loc[:, 'stat_time_day'] = pd.to_datetime(filtered_df['stat_time_day']).dt.date

In [ ]:
# Define the split logic for after 1st budget allocation
def split_budget(row):
  if pd.notna(row['total_budget']) and row['spend'] >= 0.90 * row['total_budget']:
      base = row['spend'] * (row['allocated_budget'] / row['total_budget'])
      flt = row['spend'] * (row['new_budget'] / row['total_budget'])
      rule = "Proportional"
  else:
      base = row['spend'] * 0.75
      flt = row['spend'] * 0.25
      rule = "Fixed 75/25"
  return pd.Series([base, flt, rule])

#### TikTok split

In [ ]:
# Filter for learning/initial phase dates to do 70/30 split

tt_data = filtered_df.loc[(filtered_df['stat_time_day'] >= Start_date) & (filtered_df['stat_time_day'] < date(2025, 8, 8))]
tt_data = tt_data.rename(columns={'stat_time_day': 'Date'})
tt_data = tt_data.loc[tt_data['adgroup_id'].isin(tiktok_adgroups),['adgroup_id','Date','impressions','engaged_view_15s','spend']]
tt_data = tt_data.groupby(by=['Date','adgroup_id']).sum().reset_index()
tt_data = tt_data[~((tt_data['impressions'] == 0) & (tt_data['spend'] == 0))].reset_index(drop=True)
tt_data['base_spend'] = tt_data['spend'] * 0.7
tt_data['float_spend'] = tt_data['spend'] * 0.3
tt_data.tail()

In [ ]:
# Filter for dates after the 1st change which will follow a new logic split.

tt_data_post = filtered_df.loc[(filtered_df['stat_time_day'] > date(2025, 8, 7)) & (filtered_df['stat_time_day'] < today)]
tt_data_post = tt_data_post.rename(columns={'stat_time_day': 'Date'})
tt_data_post = tt_data_post.loc[tt_data_post['adgroup_id'].isin(tiktok_adgroups),['adgroup_id','Date','impressions','engaged_view_15s','spend']]
tt_data_post = tt_data_post.groupby(by=['Date','adgroup_id']).sum().reset_index()
tt_data_post = tt_data_post[~((tt_data_post['impressions'] == 0) & (tt_data_post['spend'] == 0))].reset_index(drop=True)
tt_data_post.tail()

In [ ]:
#Convert tt_data_post data types adgroup/date
tt_data_post['adgroup_id'] = tt_data_post['adgroup_id'].astype('int64')
tt_data_post['Date'] = pd.to_datetime(tt_data_post['Date'])

In [ ]:
tt_data_post.info()

In [ ]:
# 3. Merge on both adgroup_id and date for tt and historical df
merged_tt = tt_data_post.merge(
historical_df[['adgroup_id', 'date', 'allocated_budget', 'new_budget', 'total_budget']],
left_on=['adgroup_id', 'Date'],
right_on=['adgroup_id', 'date'],
how='left'
)

In [ ]:
merged_tt.tail()

In [ ]:
# Apply and create new columns
merged_tt[['base_spend', 'float_spend', 'rule_applied']] = merged_tt.apply(split_budget, axis=1)

# Final output with impressions & completed views
tt_data_post_complete = merged_tt[['Date','adgroup_id', 'impressions', 'engaged_view_15s', 'spend', 'total_budget', 'base_spend', 'float_spend', 'rule_applied']]

tt_data_post_complete

In [ ]:
# Combine the learning phrase and post tt data

tt_data_post_complete_renamed = tt_data_post_complete[
['Date', 'adgroup_id', 'impressions', 'spend', 'engaged_view_15s', 'base_spend', 'float_spend']
]

tt_data['Date'] = pd.to_datetime(tt_data['Date'])
tt_data_post_complete_renamed['Date'] = pd.to_datetime(tt_data_post_complete_renamed['Date'])

# Step 4: Append to dv_data
tt_data_combined = pd.concat([tt_data, tt_data_post_complete_renamed], ignore_index=True)

# Optional: Sort by Date then adgroup_id
tt_data_combined = tt_data_combined.sort_values(by=['Date', 'adgroup_id']).reset_index(drop=True)
tt_data_combined.tail()

#### DV360 split

In [ ]:
# DV360 before the 1st budget change

dv_data = report_df.loc[(report_df['Date'] >= Start_date) & (report_df['Date'] < date(2025, 8, 8))]
dv_data = dv_data.loc[dv_data['Line Item ID'].isin(dv_adgroups_1),['Line Item ID','Date','Impressions','Revenue (Adv Currency)','Complete Views (Video)']]
dv_data = dv_data.rename(columns={'Line Item ID': 'adgroup_id', 'Impressions': 'impressions','Revenue (Adv Currency)': 'spend',  'Complete Views (Video)': 'completed_views'})
dv_data = dv_data.groupby(by=['Date','adgroup_id']).sum().reset_index()
dv_data = dv_data[~((dv_data['impressions'] == 0) & (dv_data['spend'] == 0))].reset_index(drop=True)
dv_data['base_spend'] = dv_data['spend'] * 0.75
dv_data['float_spend'] = dv_data['spend'] * 0.25
dv_data.tail()

In [ ]:
# using a new logic for after day 1 of budget allocation which is based how much was spend yesterday.

dv_data_post = report_df.loc[(report_df['Date'] > date(2025, 8, 7)) & (report_df['Date'] < today)]
dv_data_post = dv_data_post.loc[dv_data_post['Line Item ID'].isin(dv_adgroups_1),['Line Item ID','Date','Impressions','Revenue (Adv Currency)','Complete Views (Video)']]
dv_data_post = dv_data_post.rename(columns={'Line Item ID': 'adgroup_id', 'Impressions': 'impressions','Revenue (Adv Currency)': 'spend',  'Complete Views (Video)': 'completed_views'})
dv_data_post = dv_data_post.groupby(by=['Date','adgroup_id']).sum().reset_index()
dv_data_post = dv_data_post[~((dv_data_post['impressions'] == 0) & (dv_data_post['spend'] == 0))].reset_index(drop=True)
dv_data_post.tail()

In [ ]:
# aligned on dv360 adgroup to int
dv_data_post['adgroup_id'] = dv_data_post['adgroup_id'].astype('Int64')
dv_data_post['Date'] = pd.to_datetime(dv_data_post['Date'])

# Merge spend with budget info
# merged = dv_data_post.merge(historical_df[['adgroup_id', 'allocated_budget', 'new_budget', 'total_budget']],on='adgroup_id',how='left')

merged = dv_data_post.merge(
historical_df[['adgroup_id', 'date', 'allocated_budget', 'new_budget', 'total_budget']],
left_on=['adgroup_id', 'Date'],
right_on=['adgroup_id', 'date'],
how='left'
)

In [ ]:
# Apply and create new columns
merged[['base_spend', 'float_spend', 'rule_applied']] = merged.apply(split_budget, axis=1)

# Final output with impressions & completed views
dv_data_post_complete = merged[['Date','adgroup_id', 'impressions', 'completed_views', 'spend', 'total_budget', 'base_spend', 'float_spend', 'rule_applied']]

dv_data_post_complete

In [ ]:
dv_data_post_complete_renamed = dv_data_post_complete[['Date', 'adgroup_id', 'impressions', 'spend', 'completed_views', 'base_spend', 'float_spend']]

dv_data['Date'] = pd.to_datetime(dv_data['Date'])
# dv_data_post_complete_renamed['Date'] = pd.to_datetime(tt_data_post_complete_renamed['Date'])

# Step 4: Append to dv_data
dv_data_combined = pd.concat([dv_data, dv_data_post_complete_renamed], ignore_index=True)

# Optional: Sort by Date then adgroup_id
dv_data_combined = dv_data_combined.sort_values(by=['Date', 'adgroup_id']).reset_index(drop=True)
dv_data_combined

### Determine the base spend's pacing amount for each line item for each platform

In [ ]:
#Define the pacing function 1st

def daily_base_budget_spend(df,end_date,total_budget):

    #number of days left.
    today = dt.date.today()
    days_left = (end_date - today).days + 1

    #budget left for each day

    total_daily_budget = (total_budget-df['base_spend'].sum()) / days_left

    return round(total_daily_budget,2)



#### DV360 YouTube base allocation

In [ ]:
#DV360 YouTube base allocation
df_youtube_filtered = dv_data_combined.loc[dv_data_combined['adgroup_id'].isin(dv_adgroup_id_yt)]
df_youtube_filtered.head()

In [ ]:
Daily_base_budget_yt = daily_base_budget_spend(df_youtube_filtered,End_date,dv_base_yt_budget)
Daily_base_budget_yt

In [ ]:
df_youtube_filtered['Date'] = pd.to_datetime(df_youtube_filtered['Date'])
max_date = df_youtube_filtered['Date'].max()
df_youtube_filtered['days_ago'] = (max_date - df_youtube_filtered['Date']).dt.days

# Apply recency weights (more recent = higher weight)
df_youtube_filtered['recency_weight'] = 1 / (df_youtube_filtered['days_ago'] + 1)
df_youtube_filtered['weighted_spend'] = df_youtube_filtered['spend'] * df_youtube_filtered['recency_weight']

# Calculate averages and proportions
avg_base_spend = df_youtube_filtered.groupby('adgroup_id')['base_spend'].mean()
weighted_total = df_youtube_filtered['weighted_spend'].sum()
spend_proportions = df_youtube_filtered.groupby('adgroup_id')['weighted_spend'].sum() / weighted_total


yt_allocated__base_budget = spend_proportions * Daily_base_budget_yt

In [ ]:
yt_allocated__base_budget

#### DV360 Video Base allocation

In [ ]:
#DV360 Video base allocation

df_video_filtered = dv_data_combined.loc[dv_data_combined['adgroup_id'].isin(dv_adgroup_id_video)]
df_video_filtered.head()

In [ ]:
Daily_base_budget_video = daily_base_budget_spend(df_video_filtered,End_date,dv_base_video_budget)
Daily_base_budget_video

In [ ]:
df_video_filtered['Date'] = pd.to_datetime(df_video_filtered['Date'])
max_date = df_video_filtered['Date'].max()
df_video_filtered['days_ago'] = (max_date - df_video_filtered['Date']).dt.days

# Apply recency weights (more recent = higher weight)
df_video_filtered['recency_weight'] = 1 / (df_video_filtered['days_ago'] + 1)
df_video_filtered['weighted_spend'] = df_video_filtered['spend'] * df_video_filtered['recency_weight']

# Calculate averages and proportions
avg_base_spend = df_video_filtered.groupby('adgroup_id')['base_spend'].mean()
weighted_total = df_video_filtered['weighted_spend'].sum()
spend_proportions = df_video_filtered.groupby('adgroup_id')['weighted_spend'].sum() / weighted_total


video_allocated__base_budget = spend_proportions * Daily_base_budget_video

In [ ]:
video_allocated__base_budget

#### TikTok NU CLASS base allocation


In [ ]:
df_tiktok_nuclass_filtered = tt_data_combined.loc[tt_data_combined['adgroup_id'].isin(tiktok_adgroup_nuclass)]
df_tiktok_nuclass_filtered.head()

In [ ]:
Daily_base_budget_nuclass = daily_base_budget_spend(df_tiktok_nuclass_filtered,End_date,total_base_budget_tt/2)
Daily_base_budget_nuclass

In [ ]:
df_tiktok_nuclass_filtered['Date'] = pd.to_datetime(df_tiktok_nuclass_filtered['Date'])
max_date = df_tiktok_nuclass_filtered['Date'].max()
df_tiktok_nuclass_filtered['days_ago'] = (max_date - df_tiktok_nuclass_filtered['Date']).dt.days

# Apply recency weights (more recent = higher weight)
df_tiktok_nuclass_filtered['recency_weight'] = 1 / (df_tiktok_nuclass_filtered['days_ago'] + 1)
df_tiktok_nuclass_filtered['weighted_spend'] = df_tiktok_nuclass_filtered['spend'] * df_tiktok_nuclass_filtered['recency_weight']

# Calculate averages and proportions
avg_base_spend = df_tiktok_nuclass_filtered.groupby('adgroup_id')['base_spend'].mean()
weighted_total = df_tiktok_nuclass_filtered['weighted_spend'].sum()
spend_proportions = df_tiktok_nuclass_filtered.groupby('adgroup_id')['weighted_spend'].sum() / weighted_total


nuclass_allocated__base_budget = spend_proportions * Daily_base_budget_nuclass

In [ ]:
nuclass_allocated__base_budget

#### TikTok KOL CLASS base allocation

In [ ]:
df_tiktok_kol_filtered = tt_data_combined.loc[tt_data_combined['adgroup_id'].isin(tiktok_adgroup_kol)]
df_tiktok_kol_filtered.head()

In [ ]:
Daily_base_budget_kol = daily_base_budget_spend(df_tiktok_kol_filtered,End_date,total_base_budget_tt/2)
Daily_base_budget_kol

In [ ]:
df_tiktok_kol_filtered['Date'] = pd.to_datetime(df_tiktok_kol_filtered['Date'])
max_date = df_tiktok_kol_filtered['Date'].max()
df_tiktok_kol_filtered['days_ago'] = (max_date - df_tiktok_kol_filtered['Date']).dt.days

# Apply recency weights (more recent = higher weight)
df_tiktok_kol_filtered['recency_weight'] = 1 / (df_tiktok_kol_filtered['days_ago'] + 1)
df_tiktok_kol_filtered['weighted_spend'] = df_tiktok_kol_filtered['spend'] * df_tiktok_kol_filtered['recency_weight']

# Calculate averages and proportions
avg_base_spend = df_tiktok_kol_filtered.groupby('adgroup_id')['base_spend'].mean()
weighted_total = df_tiktok_kol_filtered['weighted_spend'].sum()
spend_proportions = df_tiktok_kol_filtered.groupby('adgroup_id')['weighted_spend'].sum() / weighted_total


kol_allocated__base_budget = spend_proportions * Daily_base_budget_kol

In [ ]:
kol_allocated__base_budget

#### Combining into one dataframe base

In [ ]:
df1 = nuclass_allocated__base_budget.reset_index()
df1.columns = ['adgroup_id', 'allocated_budget']
df2 = kol_allocated__base_budget.reset_index()
df2.columns = ['adgroup_id', 'allocated_budget']
df3 = video_allocated__base_budget.reset_index()
df3.columns = ['adgroup_id', 'allocated_budget']
df4 = yt_allocated__base_budget.reset_index()
df4.columns = ['adgroup_id', 'allocated_budget']

In [ ]:
combined__base_df = pd.concat([df1, df2,df3,df4], ignore_index=True)
combined__base_df

### Floating budget and performanc calculation

In [ ]:
# Combined the splited Tiktok and DV360 data
combined_data = pd.concat([tt_data_combined, dv_data_combined], ignore_index=True).fillna(0)
combined_data.tail(6)

In [ ]:
combined_data['float_spend'].sum()

#### Delivery capacity calcuation to be revised*

In [ ]:
# # Number of days for initial delivery capacity

n = 8

In [ ]:
learning_start = max(tt_data['Date'].min(), dv_data['Date'].min())
cutoff_date = learning_start + dt.timedelta(days=n-1)
combined_common_data = combined_data.loc[combined_data['Date'] >= learning_start,:]
learning_phase_df = combined_common_data.loc[combined_common_data['Date'] <= cutoff_date,:]
# learning_phase_max = learning_phase_df[['adgroup_id','Date','spend']].groupby(by=['adgroup_id','Date']).sum().reset_index()
# learning_phase_max = learning_phase_max[['adgroup_id','spend']].groupby(by=['adgroup_id']).max().reset_index()

In [ ]:
combined_common_data

In [ ]:
# # Delivery capacity for first day of algo

# today = dt.date.today()
# # if(cutoff_date + dt.timedelta(days=1) == today):
# #     budget_cap = learning_phase_max.copy()
# budget_cap = learning_phase_max.copy()
# if(budget_cap['spend'].sum()) < daily_overall_target:
#     gap = daily_overall_target/budget_cap['spend'].sum() + 1
#     budget_cap['spend'] = budget_cap['spend'] * gap

#### Calulate daily desired delivery

In [ ]:
def daily_float_budget_spend(df,end_date,total_budget):

    #number of days left.
    today = dt.date.today()
    days_left = (end_date - today).days + 1

    #budget left for each day

    total_daily_budget = (total_budget-df['float_spend'].sum()) / days_left

    return round(total_daily_budget,2)

In [ ]:
daily_overall_target = daily_float_budget_spend(combined_data,End_date,Total_budget)

In [ ]:
daily_overall_target

#### Each child performance calculation

In [ ]:
def calculate_cpm(dataframe, alpha):
    # Get a list of unique ad set in the dataframe
    adgroups = dataframe['adgroup_id'].unique()

    # Initialize an empty dictionary to store the results
    cpm_dict = {}

    # Loop through each ad set
    for adgroup in adgroups:
        # Filter the dataframe by ad set and sort by date in descending order #add reset index
        filtered_df = dataframe[dataframe['adgroup_id'] == adgroup].sort_values(by='Date', ascending=False)

        # Initialize variables for total cost and total clicks
        total_cost = 0
        total_imps = 0

        # Loop through each row in the filtered dataframe using enumerate()
        for i, (_, row) in enumerate(filtered_df.iterrows()):
            # Get the cost and click values for this row
            cost = row['spend']
            impressions = row['impressions']

            # Calculate the weight for this row based on its position in the dataframe
            weight = (1 - alpha) ** i

            # Add the weighted cost and clicks to the totals
            total_cost += weight * cost
            total_imps += weight * impressions

        # Calculate the cost per click using the weighted totals
        if total_imps != 0:
            cpm = total_cost / (total_imps / 1000)
        else:
            cpm = 0 #to extremely hit number

        # Add the result to the dictionary
        cpm_dict[adgroup] =cpm

    return cpm_dict

In [ ]:
def calculate_cpv(dataframe, alpha):
    # Get a list of unique ad set in the dataframe
    adgroups = dataframe['adgroup_id'].unique()

    # Initialize an empty dictionary to store the results
    cpv_dict = {}

    # Loop through each ad set
    for adgroup in adgroups:
        # Filter the dataframe by ad set and sort by date in descending order #add reset index
        filtered_df = dataframe[dataframe['adgroup_id'] == adgroup].sort_values(by='Date', ascending=False)

        # Initialize variables for total cost and total clicks
        total_cost = 0
        total_engaged_view = 0

        # Loop through each row in the filtered dataframe using enumerate()
        for i, (_, row) in enumerate(filtered_df.iterrows()):
            # Get the cost and click values for this row
            cost = row['spend']
            engaged_view = row['engaged_view_15s']

            # Calculate the weight for this row based on its position in the dataframe
            weight = (1 - alpha) ** i

            # Add the weighted cost and clicks to the totals
            total_cost += weight * cost
            total_engaged_view += weight * engaged_view

        # Calculate the cost per click using the weighted totals
        if total_engaged_view != 0:
            cpv = total_cost / total_engaged_view
        else:
            cpv = 0 #to extremely hit number

        # Add the result to the dictionary
        cpv_dict[adgroup] =cpv

    return cpv_dict

In [ ]:
def calculate_fvr(dataframe, alpha):
    # Get a list of unique ad set in the dataframe
    adgroups = dataframe['adgroup_id'].unique()

    # Initialize an empty dictionary to store the results
    fvr_dict = {}

    # Loop through each ad set
    for adgroup in adgroups:
        # Filter the dataframe by ad set and sort by date in descending order #add reset index
        filtered_df = dataframe[dataframe['adgroup_id'] == adgroup].sort_values(by='Date', ascending=False)

        # Initialize variables for total cost and total clicks
        total_imps = 0
        total_engaged_view_15s = 0

        # Loop through each row in the filtered dataframe using enumerate()
        for i, (_, row) in enumerate(filtered_df.iterrows()):
            # Get the cost and click values for this row
            impressions = row['impressions']
            engaged_view_15s = row['engaged_view_15s']

            # Calculate the weight for this row based on its position in the dataframe
            weight = (1 - alpha) ** i

            # Add the weighted cost and clicks to the totals
            total_imps += weight * impressions
            total_engaged_view_15s += weight * engaged_view_15s

        # Calculate the cost per click using the weighted totals
        if total_imps != 0:
            fvr = total_engaged_view_15s / total_imps
        else:
            fvr = 0 #to extremely hit number

        # Add the result to the dictionary
        fvr_dict[adgroup] =fvr

    return fvr_dict

In [ ]:
def calculate_vcr(dataframe, alpha):
    # Get a list of unique ad set in the dataframe
    adgroups = dataframe['adgroup_id'].unique()

    # Initialize an empty dictionary to store the results
    vcr_dict = {}

    # Loop through each ad set
    for adgroup in adgroups:
        # Filter the dataframe by ad set and sort by date in descending order #add reset index
        filtered_df = dataframe[dataframe['adgroup_id'] == adgroup].sort_values(by='Date', ascending=False)

        # Initialize variables for total cost and total clicks
        total_imps = 0
        total_video_completes = 0

        # Loop through each row in the filtered dataframe using enumerate()
        for i, (_, row) in enumerate(filtered_df.iterrows()):
            # Get the cost and click values for this row
            impressions = row['impressions']
            video_complete = row['completed_views']

            # Calculate the weight for this row based on its position in the dataframe
            weight = (1 - alpha) ** i

            # Add the weighted cost and clicks to the totals
            total_imps += weight * impressions
            total_video_completes += weight * video_complete

        # Calculate the cost per click using the weighted totals
        if total_imps != 0:
            vcr = total_video_completes / total_imps
        else:
            vcr = 0 #to extremely hit number

        # Add the result to the dictionary
        vcr_dict[adgroup] =vcr

    return vcr_dict

In [ ]:
cpm_performance = calculate_cpm(combined_data, 0.5)
vcr_performance = calculate_vcr(combined_data, 0.5)
cpv_performance = calculate_cpv(combined_data,0.5)
fvr_performance = calculate_fvr(combined_data, 0.5)

In [ ]:
cpm_performance

In [ ]:
vcr_performance

In [ ]:
cpv_performance

In [ ]:
fvr_performance

#### Comparing againts benchmarks

In [ ]:
benchmark_reference_df = combined_common_data.loc[combined_common_data['Date'] <= cutoff_date,:].copy()

benchmark_reference_df['platform'] = 'Unknown'

# Update platform based on adgroup_id
benchmark_reference_df.loc[benchmark_reference_df['adgroup_id'].isin(dv_adgroup_id_video), 'platform'] = 'DV360_video'
benchmark_reference_df.loc[benchmark_reference_df['adgroup_id'].isin(dv_adgroup_id_yt), 'platform'] = 'DV360_yt'
benchmark_reference_df.loc[benchmark_reference_df['adgroup_id'].isin(tiktok_adgroup_kol), 'platform'] = 'TikTok_kol'
benchmark_reference_df.loc[benchmark_reference_df['adgroup_id'].isin(tiktok_adgroup_nuclass), 'platform'] = 'TikTok_nuclass'



In [ ]:
benchmark_reference_df

In [ ]:
# prompt: group benchmark_reference_df by adgroup_id. do not use date column while aggregating

# Group benchmark_reference_df by adgroup_id, excluding the 'Date' column
benchmark_reference_grouped = benchmark_reference_df.groupby('platform').agg(
    {'impressions': 'sum', 'spend': 'sum', 'engaged_view_15s': 'sum', 'completed_views': 'sum'}
).reset_index()

# Calculate VCR for benchmark_reference_grouped
benchmark_reference_grouped['VCR_video'] = 0.83
benchmark_reference_grouped['VCR_yt'] = 0.88


# Calculate CPM for benchmark_reference_grouped
benchmark_reference_grouped['CPM_video'] = 40
benchmark_reference_grouped['CPM_yt'] = 50

# Calculate Cost per engage views for benchmark_reference_grouped
benchmark_reference_grouped['fvr_kol'] = 0.11
benchmark_reference_grouped['fvr_nuclass'] = 0.07


benchmark_reference_grouped

In [ ]:
# Filter for active adgroups only

# three_days_ago = today - pd.Timedelta(days=3)
# last_three_days_df = combined_data[combined_data['Date'] >= three_days_ago]
# active_li = last_three_days_df['adgroup_id'].unique()

# active_li
combined_data['Date'] = pd.to_datetime(combined_data['Date'])

# Ensure 'today' is a Pandas Timestamp
today = pd.Timestamp.today().normalize()

# Calculate threshold as Timestamp
three_days_ago = today - pd.Timedelta(days=3)

# Filter
last_three_days_df = combined_data[combined_data['Date'] >= three_days_ago]

# Get unique adgroup IDs
active_li = last_three_days_df['adgroup_id'].unique()

print(active_li)


In [ ]:
# Create a new DataFrame from vcr_performance and cpm_performance
new_df = pd.DataFrame({'adgroup_id': list(vcr_performance.keys()),
                       'vcr_performance': list(vcr_performance.values()),
                       'cpm_performance': list(cpm_performance.values()),
                       'cpv_performance': list(cpv_performance.values()),
                       'fvr_performance': list(fvr_performance.values())})

# Merge with benchmark_reference_grouped based on platform

adgroup_to_platform = {}
for adgroup_id in tiktok_adgroup_kol:
    adgroup_to_platform[adgroup_id] = 'TikTok_kol'
for adgroup_id in tiktok_adgroup_nuclass:
    adgroup_to_platform[adgroup_id] = 'TikTok_nuclass'
for adgroup_id in dv_adgroup_id_video:
    adgroup_to_platform[adgroup_id] = 'DV360_video'
for adgroup_id in dv_adgroup_id_yt:
    adgroup_to_platform[adgroup_id] = 'DV360_yt'

new_df['platform'] = new_df['adgroup_id'].map(adgroup_to_platform)


new_df = pd.merge(new_df, benchmark_reference_grouped[['platform', 'VCR_video', 'VCR_yt','CPM_video','CPM_yt','fvr_kol','fvr_nuclass']], on='platform', how='left')

# # Rename columns
new_df = new_df.rename(columns={'VCR_video': 'VCR_video_benchmark', 'VCR_yt': 'VCR_yt_benchmark','CPM_video': 'CPM_video_benchmark','CPM_yt': 'CPM_yt_benchmark','fvr_kol': 'fvr_kol_benchmark','fvr_nuclass': 'fvr_nuclass_benchmark'})

new_df = new_df[new_df['adgroup_id'].isin(active_li)].reset_index(drop=True)

# Display the resulting DataFrame
new_df


In [ ]:
new_df.replace([np.inf, -np.inf], 0, inplace=True)
new_df

In [ ]:
# Initialize column with zeros (or NaN if preferred)
new_df['vcr_index'] = 0

# Apply for YouTube group
yt_mask_1 = new_df['adgroup_id'].isin(dv_adgroup_id_yt)
new_df.loc[yt_mask_1, 'vcr_index'] = new_df.loc[yt_mask_1, 'VCR_yt_benchmark']/ new_df.loc[yt_mask_1, 'vcr_performance']

# Apply for Video group
video_mask_1 = new_df['adgroup_id'].isin(dv_adgroup_id_video)
new_df.loc[video_mask_1, 'vcr_index'] = new_df.loc[video_mask_1, 'VCR_video_benchmark']/ new_df.loc[video_mask_1, 'vcr_performance']

# Initialize column with zeros (or NaN if preferred)
new_df['fvr_index'] = 0

# Apply for tiktok kol group
tt_kol_mask = new_df['adgroup_id'].isin(tiktok_adgroup_kol)
new_df.loc[tt_kol_mask, 'fvr_index'] = new_df.loc[tt_kol_mask, 'fvr_kol_benchmark']/ new_df.loc[tt_kol_mask, 'fvr_performance']

# Apply for tiktok nuclass group
tt_nuclass_mask = new_df['adgroup_id'].isin(tiktok_adgroup_nuclass)
new_df.loc[tt_nuclass_mask, 'fvr_index'] = new_df.loc[tt_nuclass_mask, 'fvr_nuclass_benchmark']/ new_df.loc[tt_nuclass_mask, 'fvr_performance']


# Replace NaN values with 0
new_df.fillna(0, inplace=True)

new_df


In [ ]:
# Merging the two different platform index into one column
new_df['combined_index'] = new_df['vcr_index'].where(new_df['vcr_index'] != 0, new_df['fvr_index'])

In [ ]:
new_df

### Optimization

In [ ]:
import pandas as pd

# # Assuming 'combined_common_data' is the DataFrame from the previous code
# # and it has columns 'Date', 'adgroup_id', and 'spend'

# # Get today's date
# today = pd.to_datetime('today').date()

# # Calculate the date 3 days ago, excluding today
# three_days_ago = today - pd.Timedelta(days=3)

# # Filter the DataFrame to include data from the last 3 days (excluding today)
# last_three_days_data = combined_common_data[
#     (combined_common_data["Date"] >= three_days_ago) & (combined_common_data["Date"] < today)
# ]

# # Group by 'adgroup_id' and calculate the average spend
# average_spends = last_three_days_data.groupby("adgroup_id")["float_spend"].mean().reset_index()

# # Create the DataFrame
# average_float_spends_df = pd.DataFrame(average_spends)

# # Display or use the DataFrame
# average_float_spends_df
# Ensure Date column is datetime64[ns]
combined_common_data['Date'] = pd.to_datetime(combined_common_data['Date'])

# Get today's date as a Pandas Timestamp (normalized to midnight)
today = pd.Timestamp.today().normalize()

# Calculate the date 3 days ago
three_days_ago = today - pd.Timedelta(days=3)

# Filter for last 3 days excluding today
last_three_days_data = combined_common_data[
(combined_common_data["Date"] >= three_days_ago) &
(combined_common_data["Date"] < today)
]

# Group by adgroup_id and calculate average float_spend
average_spends = last_three_days_data.groupby("adgroup_id")["float_spend"].mean().reset_index()

# Create DataFrame
average_float_spends_df = pd.DataFrame(average_spends)

print(average_float_spends_df)

In [ ]:

yesterday = date.today() - timedelta(days=1)
day_before_yesterday = yesterday - timedelta(days=1)

# Filter for spends made yesterday and the day before
filtered_data = combined_common_data[
    combined_common_data["Date"].isin([day_before_yesterday])
]

# Group by 'adgroup_id' and calculate the proportion
adgroup_proportions = (last_three_days_data.groupby("adgroup_id")["float_spend"].mean()).reset_index()
adgroup_proportions = adgroup_proportions.rename(columns={"float_spend": "previous_day_spend"})
adgroup_proportions['spend_guardlines_max'] = adgroup_proportions['previous_day_spend'] * 1.5 # SIMON TO RECOMMEND
adgroup_proportions['spend_guardlines_min'] = adgroup_proportions['previous_day_spend'] * 0.5


adgroup_proportions

In [ ]:
combined_df = new_df.merge(adgroup_proportions, on='adgroup_id', how='left')
# combined_df = combined_df.rename(columns={"spend": "delivery_cap"})
combined_df = combined_df[combined_df['adgroup_id'].isin(active_li)].reset_index(drop=True)
mask_1 = combined_df['spend_guardlines_min'] == 0
combined_df.loc[mask_1, 'spend_guardlines_min'] = 5000
# Minimum budget for each child campaign - RANDOM # SIMON TO RECOMMEND

# combined_df = pd.merge(new_df, adgroup_proportions, on="adgroup_id", how="left")

# combined_df['new_min'] = combined_df[['min_budget','spend_guardlines_min']].max(axis=1)


In [ ]:
combined_df

In [ ]:
def objective_function(budget_ratios, kpi_values, regularization_strength, beta):
    kpi_values_inverted = 1 / kpi_values
    softmax_ratios = np.exp(beta * kpi_values_inverted) / np.sum(np.exp(beta * kpi_values_inverted))
    reg_term = np.sum((budget_ratios / softmax_ratios - 1)**2)
    return -np.sum(budget_ratios / kpi_values_inverted) + regularization_strength * reg_term

In [ ]:
# Inputs
parent_daily_budget = daily_overall_target
kpi_values = combined_df['combined_index'].values  # KPI values for each child campaign
# delivery_capacity = combined_df['delivery_cap'].values  # Delivery capacity for each child campaign
min_budget = combined_df['spend_guardlines_min'].values
max_budget = combined_df['spend_guardlines_max'].values  # Maximum budget for each child campaign
regularization_strength = 1  # Penalizes extreme allocations (makes sure the output we get is different from upper and lower bounds)
beta = 5 # closer to 0 = uniform distribution (even budget split)

**High Beta, Low Regularization:** This combination can lead to very concentrated allocations. The high beta makes the softmax ratios highly skewed towards the best-performing channels, and the low regularization_strength allows the optimization to allocate most of the budget to those channels.

**Low Beta, High Regularization:** This combination can lead to very even allocations. The low beta makes the softmax ratios nearly uniform, and the high regularization_strength penalizes any deviations from this uniform allocation.

**High Beta, High Regularization:** This combination can lead to a more balanced allocation, but it can also be difficult to tune. The high beta pushes the allocation towards the best-performing channels, while the high regularization_strength pulls it back towards the softmax ratios. The optimal balance depends on the specific KPI values and the desired level of concentration.

**Low Beta, Low Regularization:** This combination can lead to unpredictable results. The low beta makes the softmax ratios nearly uniform, and the low regularization_strength allows the optimization to allocate budget almost arbitrarily.

In [ ]:
# Constraints
constraints = [
    {"type": "eq", "fun": lambda x: np.sum(x) - 1},  # Sum of budget_ratios should be equal to 1
]

bounds = [(min_val / parent_daily_budget, max_val / parent_daily_budget) for min_val, max_val in zip(min_budget, max_budget)]


In [ ]:
# Initial guess
x0 = np.ones(len(kpi_values)) / len(kpi_values)

In [ ]:
# Solve the optimization problem

res = minimize(objective_function, x0, args=(kpi_values, regularization_strength, beta), bounds=bounds, constraints=constraints, method='SLSQP')

In [ ]:
# Calculate the daily budget for each child campaign
daily_budgets = res.x * parent_daily_budget

combined_df['new_budget'] = daily_budgets

In [ ]:
final_df  = combined_df[['adgroup_id','combined_index','spend_guardlines_max','spend_guardlines_min','new_budget']]
final_df

In [ ]:
final_df['new_budget'].sum()

In [ ]:
daily_overall_target

In [ ]:
# Merge the dataframes on adgroup_id
final_combined_float_base = final_df.merge(combined__base_df, on='adgroup_id', how='outer')

# Create new column with sum of allocated_budget and new_budget
final_combined_float_base['platform_total_budget'] = final_combined_float_base['allocated_budget'].fillna(0) + final_combined_float_base['new_budget'].fillna(0)

final_combined_float_base

In [ ]:
final_combined_float_base.info()

In [ ]:
# # Add today's date
final_combined_float_base['date'] = date.today()

final_combined_float_base

In [ ]:
# Make a copy so we don't modify the original
new_budget_df = final_combined_float_base.copy()

# Rename if needed
if 'platform_total_budget' in new_budget_df.columns and 'total_budget' in historical_df.columns:
  new_budget_df = new_budget_df.rename(columns={'platform_total_budget': 'total_budget'})


# Add any missing columns from historical_df
for col in historical_df.columns:
  if col not in new_budget_df.columns:
    new_budget_df[col] = np.nan  # or calculate if possible



# Reorder columns to match historical_df
new_budget_df = new_budget_df[historical_df.columns]

# Ensure both have datetime64[ns] for date
historical_df['date'] = pd.to_datetime(historical_df['date'])
new_budget_df['date'] = pd.to_datetime(new_budget_df['date'])


# Append
historical_df_new = pd.concat([historical_df, new_budget_df], ignore_index=True)


# Remove duplicates (keep the latest entry for each adgroup_id + date)
historical_df_new = historical_df_new.drop_duplicates(subset=['adgroup_id', 'date'], keep='last')

# Sort for cleanliness
historical_df_new = historical_df_new.sort_values(by=['date', 'adgroup_id']).reset_index(drop=True)

historical_df_new.tail()

In [ ]:
#SAVING NEW FILE
# Get today's date in YYYYMMDD format
today_str = datetime.today().strftime("%Y%m%d")

# Build filenames with today's date
parquet_filename = f"historical_adjustments_{today_str}.parquet"
csv_filename = f"historical_adjustments_{today_str}.csv"

# Save Parquet (safe for precision)
historical_df_new.to_parquet(parquet_filename, index=False)

# Save CSV (for sharing, with float precision control)
historical_df_new.to_csv(csv_filename, index=False, float_format="%.15g")

print(f"Saved: {parquet_filename} and {csv_filename}")

In [ ]:
# ==== CONFIG ====
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "executor_files/mp-adh-groupm-sg-cea49b7167eb.json"
BUCKET_NAME = "apac-github-test"  # Your GCS bucket name
FOLDER_PATH = "Files_for_vertex_ai/Cross_Video_Zott_historical"  # Folder inside bucket

# ==== INIT GCS CLIENT ====
client = storage.Client()

def upload_to_gcs(local_file_path, bucket_name, blob_path):
  """Uploads a file to GCS."""
  bucket = client.bucket(bucket_name)
  blob = bucket.blob(blob_path)
  blob.upload_from_filename(local_file_path)
  print(f"✅ Uploaded {local_file_path} to gs://{bucket_name}/{blob_path}")

In [ ]:
# Upload to GCS in your specified folder
upload_to_gcs(csv_filename, BUCKET_NAME, f"{FOLDER_PATH}/{csv_filename}")
upload_to_gcs(parquet_filename, BUCKET_NAME, f"{FOLDER_PATH}/{parquet_filename}")

### Update budget changes

#### DV360 CHANGES

In [ ]:
# Filter for adgroup_id in yt_adgroups and select specific columns

filtered_df_video = final_combined_float_base[final_combined_float_base['adgroup_id'].isin(dv_adgroup_id_video)][['adgroup_id', 'platform_total_budget']]
filtered_df_yt = final_combined_float_base[final_combined_float_base['adgroup_id'].isin(dv_adgroup_id_yt)][['adgroup_id', 'platform_total_budget']]

# Converting video budget
filtered_df_video['platform_total_budget'] = filtered_df_video['platform_total_budget'].astype(int)
filtered_df_video['adgroup_id'] = filtered_df_video['adgroup_id'].astype(int)
filtered_df_video = filtered_df_video.astype(str)

# Converting YT budget
filtered_df_yt['platform_total_budget'] = filtered_df_yt['platform_total_budget'].astype(int)
filtered_df_yt['adgroup_id'] = filtered_df_yt['adgroup_id'].astype(int)
filtered_df_yt = filtered_df_yt.astype(str)

In [ ]:
filtered_df_video

In [ ]:
filtered_df_yt

In [ ]:
# Inputs for Video
partner_id_input = "1278635"  # DV360 Partner ID
adv_id_input = "6993273759"
strategy_id_input = "183965"  # Put the Copilot Strategy ID
object_inputs = filtered_df_video['adgroup_id']  # List of Line Item IDs
keys = ["Pacing Amount"]
values = filtered_df_video['platform_total_budget']  # Corresponding values for each Line Item

# Generate changes for each object
changes_sdf = {object_id: dict(zip(keys, [value])) for object_id, value in zip(object_inputs, values)}

# Create JSON body
jsonbody = {
    "strategyId": strategy_id_input,
    "deltas": changes_sdf
}

# Convert to JSON string
json_ready = json.dumps(jsonbody, indent=4)
print(json_ready)

In [ ]:
# Updating through API
headers = {"developerapitoken":"e19026e0-de42-425e-913c-3e0f27baf72f", "content-type": "application/json"}

response = requests.post(url="https://optimization.choreograph.com/developerAPI/v1/SDFUpdate/LineItem", data=json_ready, headers=headers, timeout=600)
print(response.status_code)
print(response.content)
print(response.elapsed)


In [ ]:
# Inputs for YouTube
partner_id_input = "1278635"  # DV360 Partner ID
adv_id_input = "6993273759"
strategy_id_input = "183964"  # Put the Copilot Strategy ID
object_inputs = filtered_df_yt['adgroup_id']  # List of Line Item IDs
keys = ["Pacing Amount"]
values = filtered_df_yt['platform_total_budget']  # Corresponding values for each Line Item

# Generate changes for each object
changes_sdf = {object_id: dict(zip(keys, [value])) for object_id, value in zip(object_inputs, values)}

# Create JSON body
jsonbody = {
    "strategyId": strategy_id_input,
    "deltas": changes_sdf
}

# Convert to JSON string
json_ready = json.dumps(jsonbody, indent=4)
print(json_ready)

In [ ]:
# Updating through API
headers = {"developerapitoken":"e19026e0-de42-425e-913c-3e0f27baf72f", "content-type": "application/json"}

response = requests.post(url="https://optimization.choreograph.com/developerAPI/v1/SDFUpdate/LineItem", data=json_ready, headers=headers, timeout=600)
print(response.status_code)
print(response.content)
print(response.elapsed)

#### TikTok changes

In [ ]:
PATH = "/open_api/v1.3/adgroup/budget/update/"

def build_url(path, query=""):
    # type: (str, str) -> str
    """
    Build request URL
    :param path: Request path
    :param query: Querystring
    :return: Request URL
    """
    scheme, netloc = "https", "business-api.tiktok.com"
    return urlunparse((scheme, netloc, path, "", query, ""))

def post(json_str):
    # type: (str) -> dict
    """
    Send POST request
    :param json_str: Args in JSON format
    :return: Response in JSON format
    """
    url = build_url(PATH)
    args = json.loads(json_str)
    headers = {
        "Access-Token": ACCESS_TOKEN,
        "Content-Type": "application/json",
    }
    rsp = requests.post(url, headers=headers, json=args)
    return rsp.json()



In [ ]:
#kol

advertiser_id = 7371003546684424193
budget_df = final_combined_float_base.loc[final_combined_float_base['adgroup_id'].isin(tiktok_adgroup_kol),['adgroup_id','platform_total_budget']]
#Converting adgroup to str and budget to 2sf
budget_df['adgroup_id'] = budget_df['adgroup_id'].apply(lambda x: str(int(x)))
budget_df['platform_total_budget'] = budget_df['platform_total_budget'].astype(int)
campaign_id = 1839054793361409
budget_df

In [ ]:
all_args = []

for index,row in budget_df.iterrows():
  budget_value = row['platform_total_budget']
  adgroup_id = row['adgroup_id']

  # Construct the payload as a Python dictionary
  my_args = {
      "advertiser_id": str(advertiser_id),
      "budget": [
          {
              "adgroup_id": str(adgroup_id),
              "budget": budget_value
          }
      ]
  }
  all_args.append(my_args)

# Iterate through the list of payloads and make the POST request for each
for args in all_args:
  # Use json.dumps to convert the dictionary to a JSON string
  print(post(json.dumps(args)))

In [ ]:
#nuclass

advertiser_id = 7371003546684424193
budget_df = final_combined_float_base.loc[final_combined_float_base['adgroup_id'].isin(tiktok_adgroup_nuclass),['adgroup_id','platform_total_budget']]
#Converting adgroup to str and budget to 2sf
budget_df['adgroup_id'] = budget_df['adgroup_id'].apply(lambda x: str(int(x)))
budget_df['platform_total_budget'] = budget_df['platform_total_budget'].astype(int)
campaign_id = 1839054926886305
budget_df

In [ ]:
all_args = []

for index,row in budget_df.iterrows():
  budget_value = row['platform_total_budget']
  adgroup_id = row['adgroup_id']

  # Construct the payload as a Python dictionary
  my_args = {
      "advertiser_id": str(advertiser_id),
      "budget": [
          {
              "adgroup_id": str(adgroup_id),
              "budget": budget_value
          }
      ]
  }
  all_args.append(my_args)

# Iterate through the list of payloads and make the POST request for each
for args in all_args:
  # Use json.dumps to convert the dictionary to a JSON string
  print(post(json.dumps(args)))